# Camargo AI — teste de fine-tune no Colab (T4 grátis)

Treina um LoRA sobre o **Qwen3.5-4B** (mesma família do `qwen3.5:4b` que o app usa via Ollama)
com os 171 exemplos do `dataset/`, compara antes × depois, e exporta um GGUF pronto pro Ollama.

**Antes de rodar:**
1. `Ambiente de execução → Alterar tipo de ambiente → GPU T4`
2. Tenha em mãos o `dataset_openai.jsonl` (gerado no PC com `python finetune.py --export-openai dataset_openai.jsonl`) — a célula 3 pede o upload.
3. Rode as células em ordem. Treino: ~10–20 min. Export GGUF: ~10 min.

**Expectativa realista:** 171 exemplos ajustam *estilo e comportamento* (tom, formato, recusas),
não conhecimento factual — leis continuam no `knowledge/` (RAG).

In [ ]:
%%capture
# Instala Unsloth (traz trl, transformers, datasets, accelerate compatíveis)
!pip install unsloth

In [ ]:
# ---------------- Configuração ----------------
BASE_MODEL = "unsloth/Qwen3.5-4B"            # mesma família do qwen3.5:4b do app
# BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"  # alternativa: família do llama3.2:3b (mais leve/rápida)

MAX_SEQ_LENGTH = 2048
EPOCHS = 3
LEARNING_RATE = 2e-4   # padrão p/ LoRA; o 1e-5 do finetune.py é conservador demais p/ um teste curto
BATCH_SIZE = 2
GRAD_ACCUM = 4

In [ ]:
# ---------------- Upload do dataset ----------------
import json, pathlib

if not pathlib.Path("dataset_openai.jsonl").exists():
    from google.colab import files
    files.upload()  # selecione o dataset_openai.jsonl

records = [
    json.loads(l)
    for l in open("dataset_openai.jsonl", encoding="utf-8")
    if l.strip()
]
print(f"{len(records)} exemplos carregados")

In [ ]:
# ---------------- Carrega o modelo base (4-bit) ----------------
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

In [ ]:
# ---------------- Baseline: como o modelo responde ANTES ----------------
TEST_PROMPTS = [
    "O que é desvio de função no serviço público?",
    "Revise para um e-mail profissional: 'me manda os documento até sexta sem falta'",
    "Qual a capital da Austrália? E quantos habitantes ela tinha em 1830?",
]

def gerar(prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
antes = {}
for p in TEST_PROMPTS:
    antes[p] = gerar(p)
    print("=" * 80)
    print(f"PERGUNTA: {p}\n\nANTES:\n{antes[p]}\n")
FastLanguageModel.for_training(model)

In [ ]:
# ---------------- LoRA + treino ----------------
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

def to_text(rec):
    return tokenizer.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )

dataset = Dataset.from_dict({"text": [to_text(r) for r in records]})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.03,
        logging_steps=5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir="out",
        report_to="none",
    ),
)

trainer.train()

In [ ]:
# ---------------- Comparação: DEPOIS do treino ----------------
FastLanguageModel.for_inference(model)
for p in TEST_PROMPTS:
    depois = gerar(p)
    print("=" * 80)
    print(f"PERGUNTA: {p}\n")
    print(f"ANTES:\n{antes[p]}\n")
    print(f"DEPOIS:\n{depois}\n")

In [ ]:
# ---------------- Exporta GGUF q4_k_m (formato do Ollama) ----------------
model.save_pretrained_gguf("camargo_gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh camargo_gguf/*.gguf

In [ ]:
# ---------------- Salva no Google Drive (~2,5 GB) ----------------
from google.colab import drive
drive.mount("/content/drive")
!cp camargo_gguf/*q4_k_m*.gguf /content/drive/MyDrive/camargo-ft-q4_k_m.gguf
print("Salvo em MyDrive/camargo-ft-q4_k_m.gguf — baixe pelo Drive no PC")

## De volta no PC — importar no Ollama

```bash
cd ~/Downloads   # onde salvou o camargo-ft-q4_k_m.gguf

# Herda TEMPLATE e PARAMETERs do modelo base já instalado:
ollama show qwen3.5:4b --modelfile > Modelfile
# Edite o Modelfile: troque a linha FROM por:
#   FROM ./camargo-ft-q4_k_m.gguf

ollama create camargo-ft -f Modelfile
ollama run camargo-ft "teste rápido"
```

(Se treinou com `Llama-3.2-3B-Instruct`, use `ollama show llama3.2:3b --modelfile`.)

O `camargo-ft` aparece automaticamente no seletor de modelos do app —
a lista vem do `/api/tags` do Ollama. Pra comparar com o modelo atual, rode o
`evaluate.py` apontando pro novo modelo.